# Scatter plot


## 1. Setup

In [ ]:
# Import required libraries.
import ee
import geemap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.metrics import r2_score

# Initialize the Earth Engine API.
ee.Initialize()

## 2. Expand MultiPoint features

In [ ]:
# Expand MultiPoint or GeometryCollection features into individual point features.
def explode_multi_points(fc):
    def explode_feature(feature):
        geom = feature.geometry()
        geom_type = geom.type()
        is_container = ee.List(['MultiPoint', 'GeometryCollection']).contains(geom_type)

        # For container geometries, create one feature per sub-geometry and copy properties.
        def container():
            geoms = ee.List(geom.geometries())
            return ee.FeatureCollection(
                geoms.map(lambda g: ee.Feature(ee.Geometry(g)).copyProperties(feature))
            )

        # For single geometries, keep the feature unchanged.
        def single():
            return ee.FeatureCollection([feature])

        return ee.FeatureCollection(ee.Algorithms.If(is_container, container(), single()))

    # Flatten the mapped collections into one output FeatureCollection.
    return ee.FeatureCollection(fc.map(explode_feature).flatten())

## 3. Load interpreter point collections

In [ ]:
# Load the "certain" interpreter point collections.
fc1_cert = ee.FeatureCollection('projects/ee-islamkm/assets/interpreter1_200pts_certain')
fc2_cert = ee.FeatureCollection('projects/ee-islamkm/assets/interpreter2_200pts_certain')
fc3_cert = ee.FeatureCollection('projects/ee-islamkm/assets/interpreter3_200pts_certain')

# Expand container geometries into individual point features.
fc1_cert = explode_multi_points(fc1_cert)
fc2_cert = explode_multi_points(fc2_cert)
fc3_cert = explode_multi_points(fc3_cert)

# Load the full interpreter point collections.
fc1_conf = ee.FeatureCollection('projects/ee-islamkm/assets/interpreter1_200pts')
fc2_conf = ee.FeatureCollection('projects/ee-islamkm/assets/interpreter2_200pts')
fc3_conf = ee.FeatureCollection('projects/ee-islamkm/assets/interpreter3_200pts')

# Tag each collection so the source can be tracked later.
fc1_cert = fc1_cert.map(lambda f: f.set('source', 'cert'))
fc1_conf = fc1_conf.map(lambda f: f.set('source', 'conf'))
fc2_cert = fc2_cert.map(lambda f: f.set('source', 'cert'))
fc2_conf = fc2_conf.map(lambda f: f.set('source', 'conf'))
fc3_cert = fc3_cert.map(lambda f: f.set('source', 'cert'))
fc3_conf = fc3_conf.map(lambda f: f.set('source', 'conf'))

# Merge the confident and full collections for each interpreter.
fc1 = fc1_cert.merge(fc1_conf)
fc2 = fc2_cert.merge(fc2_conf)
fc3 = fc3_cert.merge(fc3_conf)

## 4. Load stacked generalization model outputs

In [ ]:
# Load the model prediction rasters.
logreg_npt = ee.Image("projects/ee-ashrafulcuetbd/assets/stacking_logreg_npt_prediction")
logreg_pt = ee.Image("projects/ee-ashrafulcuetbd/assets/stacking_logreg_pt_prediction")
rf_npt = ee.Image("projects/ee-ashrafulcuetbd/assets/stacking_rf_npt_prediction")
rf_pt = ee.Image("projects/ee-ashrafulcuetbd/assets/stacking_rf_pt_prediction")

## 5. Sampling helpers and point-level statistics

In [ ]:
# Sample the mean of the first image band over a geometry.
def sample_image_mean(img, geom, scale=10):
    band = ee.String(img.bandNames().get(0))
    mean_dict = img.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=geom,
        scale=scale,
        maxPixels=1e8
    )
    return ee.Number(mean_dict.get(band))


# Compute interpreter and model statistics for each reference point.
def compute_point_stats_v2(feat):
    geom = feat.geometry()
    buf = geom.buffer(10)

    # Compute interpreter statistics within the 10 m buffer.
    i1 = ee.Number(fc1.filterBounds(buf).aggregate_mean('val_int'))
    i2 = ee.Number(fc2.filterBounds(buf).aggregate_mean('val_int'))
    i3 = ee.Number(fc3.filterBounds(buf).aggregate_mean('val_int'))

    interp_mean = i1.add(i2).add(i3).divide(3)
    interp_mean_sq = i1.pow(2).add(i2.pow(2)).add(i3.pow(2)).divide(3)
    interp_std = interp_mean_sq.subtract(interp_mean.pow(2)).max(0).sqrt()

    # Sample model predictions within the same buffer.
    logreg_npt_v = sample_image_mean(logreg_npt, buf, scale=10)
    logreg_pt_v = sample_image_mean(logreg_pt, buf, scale=10)
    rf_npt_v = sample_image_mean(rf_npt, buf, scale=10)
    rf_pt_v = sample_image_mean(rf_pt, buf, scale=10)

    # Return the feature with appended statistics.
    return feat.set({
        'source': feat.get('source'),
        'i1_mean': i1,
        'i2_mean': i2,
        'i3_mean': i3,
        'interp_std': interp_std,
        'logreg_npt_mean': logreg_npt_v,
        'logreg_pt_mean': logreg_pt_v,
        'rf_npt_mean': rf_npt_v,
        'rf_pt_mean': rf_pt_v,
    })

## 6. Compute point-level results and convert to a DataFrame

In [ ]:
# Use interpreter 1 as the reference geometry collection.
result_fc = fc1.map(compute_point_stats_v2)

# Convert the Earth Engine FeatureCollection to a pandas DataFrame.
df2 = geemap.ee_to_df(result_fc)

# Display the resulting table.
print(df2)

## 7. Plot interpreter means against model predictions

In [ ]:
# Compute the interpreter mean from the three interpreter summaries.
df2['interp_mean'] = df2[['i1_mean', 'i2_mean', 'i3_mean']].mean(axis=1)

# Define the model columns used throughout the analysis.
model_cols = ['logreg_npt_mean', 'logreg_pt_mean', 'rf_npt_mean', 'rf_pt_mean']
model_labels = [col.removesuffix('_mean') for col in model_cols]

# Create a 2 x 2 plotting grid.
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

# Plot scatter points, 1:1 lines, and linear fits for each model.
for ax, col, label in zip(axes, model_cols, model_labels):
    cert_mask = (df2['source'] == 'cert') & df2['interp_mean'].notna() & df2[col].notna()
    conf_mask = (df2['source'] == 'conf') & df2['interp_mean'].notna() & df2[col].notna()

    x_cert = df2.loc[cert_mask, 'interp_mean']
    y_cert = df2.loc[cert_mask, col]
    x_conf = df2.loc[conf_mask, 'interp_mean']
    y_conf = df2.loc[conf_mask, col]

    ax.scatter(
        x_cert,
        y_cert,
        alpha=0.7,
        s=40,
        edgecolor='k',
        color='steelblue',
        label='Base-learners agree'
    )
    ax.scatter(
        x_conf,
        y_conf,
        alpha=0.7,
        s=40,
        edgecolor='k',
        color='coral',
        label='Less consensus among base-learners'
    )

    x_all = df2.loc[cert_mask | conf_mask, 'interp_mean']
    y_all = df2.loc[cert_mask | conf_mask, col]

    line_min = min(x_all.min(), y_all.min())
    line_max = max(x_all.max(), y_all.max())
    ax.plot([line_min, line_max], [line_min, line_max], 'r--', label='1:1 line')

    coeffs = np.polyfit(x_all, y_all, deg=1)
    poly = np.poly1d(coeffs)
    x_fit = np.linspace(line_min, line_max, 100)
    ax.plot(x_fit, poly(x_fit), 'b-', label='Linear fit')

    y_pred = poly(x_all)
    r2 = r2_score(y_all, y_pred)

    ax.set_title(label)
    ax.set_xlabel('Interpreter mean')
    ax.set_ylabel('Model mean')
    ax.text(0.05, 0.95, f'R² = {r2:.2f}', transform=ax.transAxes, va='top')
    ax.legend()

plt.tight_layout()
plt.show()

## 8. Goodness-of-fit test

In [ ]:
# Fit an OLS model for each prediction raster against interpreter mean.
results = {}

for col in model_cols:
    x = df2['interp_mean']
    y = df2[col]

    mask = x.notna() & y.notna()
    x_clean = x[mask]
    y_clean = y[mask]

    X = sm.add_constant(x_clean)
    model = sm.OLS(y_clean, X).fit()

    print("\n" + "=" * 80)
    print(f"OLS summary for: {col}  (n={len(y_clean)})")
    print("=" * 80)
    print(model.summary())

    results[col] = {
        'model': model,
        'rsquared': model.rsquared,
        'adj_rsquared': model.rsquared_adj,
        'f_pvalue': model.f_pvalue,
        'params': model.params,
        'nobs': int(model.nobs)
    }

We conducted ordinary least squares (OLS) regression to test how well interpreter predictions match actual model predictions, and all four models showed statistically significant fits (p < 0.001) with R² values between 0.45-0.54. The regression slopes near 1.0 and intercepts close to 0 indicate good calibration—meaning the interpreters not only correlate well with the models but also predict similar magnitudes of output values.

## 9. Confusion matrices

In [ ]:
# Recode interpreter mean into three categories.
conditions = [
    df2['interp_mean'] <= 0.3,
    df2['interp_mean'] >= 0.7
]
choices = ['0.3 or less', '0.7 or more']

df2['interp_mean_cat'] = np.select(
    conditions,
    choices,
    default='confused_points_0.3to0.7'
)

# Build confusion matrices for each model.
conf_mats = {}

for col in model_cols:
    conditions_model = [
        df2[col] <= 0.3,
        df2[col] >= 0.7
    ]

    df2[f'{col}_cat'] = np.select(
        conditions_model,
        choices,
        default='confused_points_0.3to0.7'
    )

    cm = pd.crosstab(
        df2['interp_mean_cat'],
        df2[f'{col}_cat'],
        rownames=['Interpreter'],
        colnames=[col]
    )

    cm = cm.reindex(
        index=['0.3 or less', 'confused_points_0.3to0.7', '0.7 or more'],
        columns=['0.3 or less', 'confused_points_0.3to0.7', '0.7 or more'],
        fill_value=0
    )

    conf_mats[col] = cm

# Print confusion matrices.
for col, cm in conf_mats.items():
    print(f"\nConfusion matrix for {col}:\n")
    print(cm)

## 10. Accuracy metrics from confusion matrices

In [ ]:
# Compute overall, producer's, and user's accuracy from a confusion matrix.
def accuracy_metrics(cm):
    oa = cm.values.diagonal().sum() / cm.values.sum() * 100
    pa = cm.apply(lambda row: row[row.name] / row.sum() * 100, axis=1)
    ua = cm.apply(lambda col: col[col.name] / col.sum() * 100, axis=0)

    return {
        'Overall Accuracy (%)': oa,
        'Producer Accuracy (%)': pa,
        'User Accuracy (%)': ua
    }


# Print accuracy summaries for each model.
for col, cm in conf_mats.items():
    metrics = accuracy_metrics(cm)

    print(f"\nAccuracy metrics for {col}:")
    print(f"Overall Accuracy: {metrics['Overall Accuracy (%)']:.2f}%")
    print("Producer Accuracy (%):")
    print(metrics['Producer Accuracy (%)'].round(2))
    print("User Accuracy (%):")
    print(metrics['User Accuracy (%)'].round(2))

## 11. Check the number of points used

In [ ]:
# Print valid point counts by model and source group.
print("Point counts by model and source:")
print("=" * 60)

for col in model_cols:
    cert_mask = (df2['source'] == 'cert') & df2['interp_mean'].notna() & df2[col].notna()
    conf_mask = (df2['source'] == 'conf') & df2['interp_mean'].notna() & df2[col].notna()

    n_cert = cert_mask.sum()
    n_conf = conf_mask.sum()
    n_total = n_cert + n_conf

    print(f"\n{col}:")
    print(f"  Base-learners agree (cert):                {n_cert:,}")
    print(f"  Less consensus among base-learners (conf): {n_conf:,}")
    print(f"  Total points:                              {n_total:,}")

print("\n" + "=" * 60)